# Prompt Claude Batch

Code for Batch Claude requests.

In [226]:
import anthropic
import json
import yaml
from pathlib import Path
from typing import Dict, Any, List
import os
import time

from output_formats import output_json_explain, output_json_no_explain

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request as BatchRequest

In [227]:
def load_secrets(file_path: str = "secrets.json") -> dict:
    """Load API secrets from JSON file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Secrets file not found: {file_path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["CLAUDE_API_KEY"]
    
api_key = load_secrets("secrets.json")
client = anthropic.Anthropic(api_key=api_key)

In [228]:
# Load prompts from file
def load_prompts(file_path: str):
    """Load system and user prompts from a JSON or YAML file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Prompt file not found: {file_path}")

    if path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    elif path.suffix in [".yaml", ".yml"]:
        with open(path, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
    else:
        raise ValueError("Unsupported file format. Use .json or .yaml")

    sys_prompt = data.get("system", "You are an expert annotator.")
    user_prompt = data["user"] 
    return sys_prompt, user_prompt

In [229]:
input_type = "manual"
output_type = "json-explain"
n_shot = "0shot"
model = "claude-3-5-haiku-20241022"
split = "test"
temperature = 1.0
max_items = None # set to an int like 20 for a quick smoke test

sys_prompt, user_prompt = load_prompts(f"../../../prompts/{input_type}_{output_type}_{n_shot}.yaml")

data_path = "../../../data/reannotated_20250509_all.json"
results_path = f"../../../results/{input_type}_{output_type}_{n_shot}_{model}_{split}.json"

with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

# Filter data where 'split' == split
filtered_data = {k: v for k, v in data.items() if v.get('split') == split}
number_examples = len(filtered_data)


In [230]:
with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

# Filter data where 'split' == split
filtered_data = {k: v for k, v in data.items() if v.get('split') == split}
number_examples = len(filtered_data)

In [231]:
# Build sample list and respect optional cap
samples = list(filtered_data.items())
try:
    max_items  # may be defined above
except NameError:
    max_items = None
if max_items is not None:
    samples = samples[:max_items]

In [232]:
# Load existing partial results to support resume/merge
existing_by_id: Dict[str, Dict[str, Any]] = {}
if os.path.exists(results_path) and os.path.getsize(results_path) > 0:
    try:
        with open(results_path, "r", encoding="utf-8") as f:
            prior = json.load(f)
            if isinstance(prior, dict):
                prior = [prior]
            for row in prior:
                tid = row.get("text_id")
                if tid is not None:
                    existing_by_id[str(tid)] = row
        print(
            f"Loaded {len(existing_by_id)} existing results; new batch will skip those."
        )
    except Exception as e:
        print(f"Warning: could not read existing results: {e}")

In [233]:
# Build Batch API requests (skip empties and already-completed)
requests: List[BatchRequest] = []
empty_entries: List[Dict[str, Any]] = []

for key, item in samples:
    k = str(key)
    if k in existing_by_id:
        continue
    text = (item.get("text") or "").strip()
    if not text:
        empty_entries.append(
            {
                "text_id": k,
                "input_text": text,
                "split": split,
                "output": {"error": "empty_text"},
            }
        )
        continue
    user_content = user_prompt.format(text=text)

    params = MessageCreateParamsNonStreaming(
        model=model,
        max_tokens=1500,
        temperature=temperature,
        system=[{"type": "text", "text": sys_prompt}],
        messages=[{"role": "user", "content": user_content}],
        tools=[output_json_explain] if "explain" in output_type else [output_json_no_explain],
        tool_choice={"type": "tool", "name": "MoralisierungOutput"}
    )
    req = BatchRequest(custom_id=k, params=params)
    requests.append(req)

print(
    f"Prepared {len(requests)} Batch API requests; {len(empty_entries)} empty texts skipped."
)

Prepared 1587 Batch API requests; 0 empty texts skipped.


In [234]:
# Submit batch
batch = client.messages.batches.create(requests=requests)
batch_id = getattr(batch, "id", None) or getattr(batch, "batch_id", None)
print(f"Created batch: {batch_id}")

def _get_state(obj):
    return (
        getattr(obj, "processing_status", None)
        or getattr(obj, "status", None)
        or getattr(obj, "state", None)
        or ""
    )

Created batch: msgbatch_01DoHEM2crGwciiVEFjtHocL


In [235]:
# Poll until completed or failed
seen_state = None
while True:
    state = str(_get_state(batch)).lower()
    if state != seen_state:
        print(f"Batch state: {state}")
        seen_state = state
    if state == 'ended':
        break
    if state in ("failed", "errored", "error", "cancelled", "canceled"):
        raise RuntimeError(f"Batch failed: {batch}")
    time.sleep(5)
    batch = client.messages.batches.retrieve(batch_id)

Batch state: in_progress
Batch state: ended


In [236]:
# Helper: extract first tool_use block input from Anthropic response
def extract_tool_output(message):
    for block in getattr(message, "content", []):
        if getattr(block, "type", "") == "tool_use":
            # Claude tool call: block.name, block.input
            return getattr(block, "input", None)

entries: List[Dict[str, Any]] = []

# Stream results file in memory-efficient chunks, processing one at a time
for result in client.messages.batches.results(
    batch_id,
):
    match result.result.type:
        case "succeeded":
            entry = {
                "text_id": result.custom_id,
                "input_text": filtered_data[result.custom_id]['text'],
                "split": split,
                "output": extract_tool_output(result.result.message)
            }
            entries.append(entry)
        case "errored":
            if result.result.error.type == "invalid_request":
                # Request body must be fixed before re-sending request
                print(f"Validation error {result.custom_id}")
            else:
                # Request can be retried directly
                print(f"Server error {result.custom_id}")
        case "expired":
            print(f"Request expired {result.custom_id}")

# Write results to results_path
with open(results_path, "w", encoding="utf-8") as f:
    if existing_by_id:
        json.dump([existing_by_id] + entries, f, ensure_ascii=False, indent=2) # Save existing partial results and new results
        print(f"{len([existing_by_id] + entries)} results saved to {results_path}!")
    else:
        json.dump(entries, f, ensure_ascii=False, indent=2)
        print(f"{len(entries)} results saved to {results_path}!")

1587 results saved to ../../../results/manual_json-explain_0shot_claude-3-5-haiku-20241022_test.json!
